In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PS3 — Root Cause Classification
# SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor
#
# Architecture  : CUBIC MARS AWS v6
# Targets       : failure_level_label (3-class: MINOR/MAJOR/CRITICAL)
#                 root_cause_category (8-class: rule-based text derivation)
# Grain         : one row per ServiceNow availability event per device
# Models        : XGBoost multiclass + LightGBM + CatBoost
#                 NLP pipeline (TF-IDF on fault text) + Combined structured+NLP
# Feasibility   : 70%  (30% gap — SVN_STAGE=0 rows; AE_FAILURE_LEVEL Ventra taxonomy confirmed 2026-06-23)
#                 AE_FAILURE_LEVEL (hardware faults): 1=NONPAYMENT, 2=PURCHASE_CARD,
#                   3=PURCHASE_PRODUCT, 4=ALL_PURCHASE, 5=ALL_FUNCTIONS, 16=BUS_READER_ASSEMBLY
#                 Decoded via silver.dim_failure_level (S18). Updated from 55% → 70%.
# Data source   : CTA.SERVICENOW_AVAILABILITY_EVENTS (295,960 rows) — simplified target
# ═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# ── CELL 1 : Install ──────────────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs","pyarrow",
    "scikit-learn>=1.4","xgboost>=2.0","lightgbm>=4.3","catboost>=1.2",
    "optuna>=3.6","shap>=0.45","mlflow>=2.13", "sagemaker-mlflow",
    "boto3","sagemaker>=2.220,<3.0","pandas>=2.0","matplotlib","seaborn","nltk",
]
subprocess.check_call([sys.executable,"-m","pip","install","-q","--upgrade"]+pkgs)
import nltk; nltk.download("stopwords",quiet=True); nltk.download("wordnet",quiet=True)
print("OK")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────
import os, json, time, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile

# deltalake removed: gold tables exported as Parquet; pd.read_parquet() used in CELL 4
from sklearn.model_selection         import TimeSeriesSplit, StratifiedKFold
from sklearn.preprocessing           import LabelEncoder, label_binarize
from sklearn.pipeline                import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model            import LogisticRegression
from sklearn.ensemble                import RandomForestClassifier
from sklearn.base                    import BaseEstimator, TransformerMixin
from sklearn.metrics                 import (roc_auc_score, f1_score,
                                             classification_report,
                                             confusion_matrix)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier, early_stopping, log_evaluation
from catboost  import CatBoostClassifier

import mlflow, mlflow.sklearn
from mlflow.models.signature import infer_signature
try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session                       # sagemaker >= 3
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role            # sagemaker >= 3
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Imports OK")

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────────
ARTIFACT_BUCKET     = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
GOLD_BUCKET         = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX         = "chicago/gold/device_ps3_incident"
GOLD_S3             = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION          = Session()
ROLE                = get_execution_role()
REGION              = boto3.Session().region_name
SM_CLIENT           = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT           = boto3.client("s3", region_name=REGION)

MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback (EFS)")
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)  # sagemaker-mlflow resolves arn: scheme

EXPERIMENT_NAME     = "chicago-ps3-root-cause"
MLFLOW_MODEL_NAME   = "ps3-root-cause-classification"
MODEL_REGISTRY_NAME = "chicago-ps3-root-cause"
FS_GROUP_NAME       = "chicago-ps3-features"
ENDPOINT_NAME       = "chicago-ps3-rootcause-v1"

# Primary target: 3-class failure level (available via CTA mirror)
# Original target (9-class fault codes) BLOCKED — SVN_STAGE.U_FS_FAULT_CODES = 0 rows
# AE_FAILURE_LEVEL Ventra taxonomy confirmed 2026-06-23 (feasibility updated 55% → 70%)
TARGET          = "failure_level_label"    # MINOR=0 / MAJOR=1 / CRITICAL=2
TARGET_NUMERIC  = "failure_level"          # 1/2/3 → encoded to 0/1/2
NUM_CLASSES     = 3
CLASS_NAMES     = ["MINOR","MAJOR","CRITICAL"]

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Gold S3: {GOLD_S3}")
print("NOTE: PS3 feasibility=70% (updated 2026-06-23). SVN_STAGE.U_FS_FAULT_CODES/ACTION_CODES=0 rows.")
print("      9-class fc_description BLOCKED. Using 3-class failure_level_label.")

In [ ]:
# ── CELL 4 : Load Gold table (Parquet on S3) ──────────────────────────────────
print(f"Loading Gold table from: {GOLD_S3}")
df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

df["event_date"] = pd.to_datetime(df["event_date"])

# Encode failure_level 1/2/3 → 0/1/2 (MINOR/MAJOR/CRITICAL)
if TARGET_NUMERIC in df.columns:
    df[TARGET] = df[TARGET_NUMERIC].fillna(1).astype(int) - 1
    df[TARGET] = df[TARGET].clip(0, NUM_CLASSES - 1)

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['event_date'].min().date()}  →  {df['event_date'].max().date()}")
print(f"\nTarget distribution (0=MINOR, 1=MAJOR, 2=CRITICAL):\n{df[TARGET].value_counts()}")
print(f"\nData source: {DATA_SOURCE}  |  Gold version: {GOLD_VERSION}")

In [ ]:
# ── CELL 5 : Rule-based root_cause_category (8-class) ───────────────────────
# Derived from NLP text match on AE_FAULT_DESCRIPTION / AE_SYMPTOM.
# No ML model needed — deterministic rule engine.
# This is the workaround for SVN_STAGE.U_FS_ACTION_CODES = 0 rows.

CATEGORIES = [
    ("CASH_HANDLING", ["cash","note","coin","jam","dispense","banknote","bna","ctu"]),
    ("PRINTER",       ["print","paper","receipt","roll","thermal"]),
    ("CARD_READER",   ["card","chip","swipe","reader","csc","nfc","contactless"]),
    ("GATE_MECH",     ["gate","arm","barrier","motor","turnstile"]),
    ("COMMS",         ["comms","network","timeout","disconnect","connect","wifi","tcp"]),
    ("POWER",         ["power","voltage","ups","battery","surge","trip"]),
    ("SOFTWARE",      ["software","firmware","crash","reboot","update","exception"]),
    ("UNKNOWN",       []),   # catch-all
]

def classify_root_cause(text):
    if pd.isna(text):
        return "UNKNOWN"
    t = str(text).lower()
    for cat, keywords in CATEGORIES[:-1]:
        if any(k in t for k in keywords):
            return cat
    return "UNKNOWN"

text_cols = [c for c in ["AE_FAULT_DESCRIPTION","AE_SYMPTOM","AE_PROBLEM"]
             if c in df.columns]
if text_cols:
    combined_text = df[text_cols].fillna("").agg(" ".join, axis=1)
    df["root_cause_category"] = combined_text.apply(classify_root_cause)
    print(f"\nRoot cause distribution:\n{df['root_cause_category'].value_counts()}")

    # Log rule stats to MLflow
    with mlflow.start_run(run_name="rule_based_root_cause") as run:
        mlflow.set_tags({"ps":"PS3","model_type":"rule_based","city":"chicago"})
        dist = df["root_cause_category"].value_counts()
        for cat, count in dist.items():
            mlflow.log_metric(f"cat_{cat}", int(count))
        mlflow.log_param("n_categories",len(CATEGORIES))
        mlflow.log_param("n_text_cols",len(text_cols))
        mlflow.log_param("coverage",
            round((df["root_cause_category"]!="UNKNOWN").mean(),4))
        # Save rule config
        rule_config = {c:kw for c,kw in CATEGORIES}
        with open("/tmp/root_cause_rules.json","w") as f:
            json.dump(rule_config,f,indent=2)
        mlflow.log_artifact("/tmp/root_cause_rules.json","rules")
        RULES_RUN = run.info.run_id
    print(f"\nRule-based categories logged. Coverage: {(df['root_cause_category']!='UNKNOWN').mean():.1%}")
else:
    print("No text columns in Gold — rule-based root cause skipped")

In [ ]:
# ── CELL 6 : EDA ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

pd.Series(CLASS_NAMES)[df[TARGET].value_counts().sort_index().index].pipe(
    lambda x: df[TARGET].value_counts().sort_index()).plot(
    kind="bar", ax=axes[0], color=["#43A047","#F57F17","#C62828"], edgecolor="white")
axes[0].set_xticklabels(CLASS_NAMES, rotation=0)
axes[0].set_title("Failure Level Distribution")

if text_cols:
    df["root_cause_category"].value_counts().plot(kind="barh",ax=axes[1],color="#1565C0")
    axes[1].set_title("Root Cause Category (rule-based)")

df.groupby("event_date")[TARGET].value_counts().unstack(fill_value=0).plot(
    ax=axes[2], linewidth=0.7)
axes[2].set_title("Failure level over time")

plt.tight_layout(); plt.savefig("/tmp/ps3_eda.png",dpi=100); plt.show()

In [ ]:
# ── CELL 7 : Feature engineering ─────────────────────────────────────────────
TARGET_COL  = TARGET
ID_COLS     = ["DEVICE_ID","event_date","split","AE_FAULT_DESCRIPTION",
               "AE_SYMPTOM","AE_PROBLEM","AE_RESOLUTION",
               "root_cause_category","failure_level","failure_level_label"]
ID_COLS     = [c for c in ID_COLS if c in df.columns]

# Structured numeric features
STRUCT_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET_COL]
    and pd.api.types.is_numeric_dtype(df[c])
]
df[STRUCT_COLS] = df[STRUCT_COLS].fillna(df[STRUCT_COLS].median())

# Text features: combine available NLP columns
if text_cols:
    df["combined_text"] = df[text_cols].fillna("").agg(" ".join, axis=1)
    df["combined_text"] = df["combined_text"].str.lower().str.replace(
        r"[^a-z\s]","",regex=True)

print(f"Structured features : {len(STRUCT_COLS)}")
print(f"Text features       : {len(text_cols)} columns → combined_text")

# Temporal splits (by event_date)
train_df = df[df["event_date"] <  "2025-10-01"]
val_df   = df[(df["event_date"] >= "2025-10-01") & (df["event_date"] < "2026-01-01")]
test_df  = df[df["event_date"] >= "2026-01-01"]

X_train_s = train_df[STRUCT_COLS].values; y_train = train_df[TARGET_COL].values
X_val_s   = val_df[STRUCT_COLS].values;   y_val   = val_df[TARGET_COL].values
X_test_s  = test_df[STRUCT_COLS].values;  y_test  = test_df[TARGET_COL].values
X_cv      = np.vstack([X_train_s,X_val_s])
y_cv      = np.concatenate([y_train,y_val])

print(f"Train:{len(X_train_s):,}  Val:{len(X_val_s):,}  Test:{len(X_test_s):,}")

In [ ]:
# ── CELL 8 : Feature Store ────────────────────────────────────────────────────
fs_df = train_df[STRUCT_COLS+[TARGET_COL,"DEVICE_ID","event_date"]].copy()
fs_df["event_time"] = fs_df["event_date"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
fs_df["DEVICE_ID"]  = fs_df["DEVICE_ID"].astype(str)
fs_df               = fs_df.drop(columns=["event_date"])

def dtype_to_fs(dtype):
    if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
    if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
    return FeatureTypeEnum.STRING

feature_group=FeatureGroup(name=FS_GROUP_NAME,sagemaker_session=SM_SESSION,
    feature_definitions=[
        FeatureDefinition(feature_name=c,feature_type=dtype_to_fs(fs_df[c].dtype))
        for c in fs_df.columns])
try:
    feature_group.create(
        s3_uri=f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
        record_identifier_name="DEVICE_ID",event_time_feature_name="event_time",
        role_arn=ROLE,enable_online_store=True,
        description="PS3 root cause classification features — Chicago Ventra")
    status=""
    while status!="Created":
        time.sleep(15)
        status=feature_group.describe().get("FeatureGroupStatus")
        print(f"  {status}")
except Exception as e: print(f"Feature Group: {e}")

feature_group.ingest(data_frame=fs_df.reset_index(drop=True),max_workers=4,wait=True)
print(f"Ingested {len(fs_df):,} → {FS_GROUP_NAME}")

In [ ]:
# ── CELL 9 : Helpers ──────────────────────────────────────────────────────────
COMMON_TAGS = {"ps":"PS3","target":"failure_level_label (3-class)","city":"chicago",
               "gold_version":str(GOLD_VERSION),"feasibility":"70pct",
               "architecture":"CUBIC-MARS-AWS-v6"}

def evaluate_mc(model, X, y, split="test"):
    prob = model.predict_proba(X)
    pred = prob.argmax(axis=1)
    # Macro AUC (one-vs-rest)
    try:
        auc = roc_auc_score(y, prob, multi_class="ovr", average="macro")
    except: auc = 0.0
    return {
        f"{split}_auc_macro" : round(auc, 4),
        f"{split}_f1_macro"  : round(f1_score(y, pred, average="macro", zero_division=0), 4),
        f"{split}_f1_weighted": round(f1_score(y, pred, average="weighted", zero_division=0), 4),
    }

def log_cm_mc(model, X, y, name):
    pred = model.predict_proba(X).argmax(axis=1)
    cm   = confusion_matrix(y, pred)
    fig,ax=plt.subplots(figsize=(5,4))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=ax,
                xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES)
    ax.set_title(f"{name} — confusion matrix")
    fig.tight_layout(); fig.savefig(f"/tmp/cm_{name}.png",dpi=80); plt.close(fig)
    mlflow.log_artifact(f"/tmp/cm_{name}.png")

In [ ]:
# ── CELL 10 : MODEL 1 — NLP Pipeline (TF-IDF + Logistic Regression) ──────────
# Structured NLP baseline — uses only text features.
# Establishes how much information is in fault description text alone.

if text_cols and "combined_text" in df.columns:
    with mlflow.start_run(run_name="tfidf_logistic") as run:
        mlflow.set_tags({**COMMON_TAGS,"model_type":"nlp_tfidf_lr"})

        nlp_pipe = Pipeline([
            ("tfidf", TfidfVectorizer(
                max_features  = 5000,
                ngram_range   = (1, 2),
                min_df        = 3,
                strip_accents = "unicode",
                sublinear_tf  = True,
            )),
            ("clf", LogisticRegression(
                C             = 1.0,
                multi_class   = "multinomial",
                class_weight  = "balanced",
                max_iter      = 1000,
                random_state  = 42,
            )),
        ])

        nlp_pipe.fit(train_df["combined_text"], y_train)

        # Wrap for predict_proba interface consistency
        class NLPWrapper:
            def __init__(self, pipe): self.pipe = pipe
            def predict_proba(self, X): return self.pipe.predict_proba(X)
            def predict(self, X): return self.pipe.predict(X)

        vm = evaluate_mc(nlp_pipe, val_df["combined_text"],  y_val,  "val")
        tm = evaluate_mc(nlp_pipe, test_df["combined_text"], y_test, "test")
        mlflow.log_params({"max_features":5000,"ngram_range":"(1,2)","C":1.0})
        mlflow.log_metrics({**vm,**tm})
        log_cm_mc(nlp_pipe, test_df["combined_text"], y_test, "tfidf_lr")

        sig = infer_signature(train_df["combined_text"].values[:3],
                               nlp_pipe.predict_proba(train_df["combined_text"].values[:3]))
        mlflow.sklearn.log_model(nlp_pipe,"model",signature=sig)
        NLP_RUN = run.info.run_id

    print(f"TF-IDF+LR  val_auc={vm['val_auc_macro']}  test_auc={tm['test_auc_macro']}")
else:
    print("No text columns — NLP pipeline skipped"); NLP_RUN = None

In [ ]:
# ── CELL 11 : MODEL 2 — XGBoost multiclass (structured features) ─────────────
with mlflow.start_run(run_name="xgboost_multiclass") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"xgboost_multiclass"})

    xgb = XGBClassifier(
        n_estimators    = 500,
        max_depth       = 6,
        learning_rate   = 0.05,
        subsample       = 0.8,
        colsample_bytree= 0.8,
        objective       = "multi:softprob",
        num_class       = NUM_CLASSES,
        eval_metric     = "mlogloss",
        early_stopping_rounds = 40,
        tree_method     = "hist",
        random_state    = 42,
        n_jobs          = -1,
    )
    xgb.fit(X_train_s, y_train, eval_set=[(X_val_s,y_val)], verbose=100)

    vm = evaluate_mc(xgb, X_val_s, y_val, "val")
    tm = evaluate_mc(xgb, X_test_s, y_test, "test")
    mlflow.log_params({"n_estimators":500,"max_depth":6,"num_class":NUM_CLASSES,
                       "best_iter":xgb.best_iteration})
    mlflow.log_metrics({**vm,**tm})
    log_cm_mc(xgb, X_test_s, y_test, "xgb_mc")

    sig=infer_signature(X_train_s,xgb.predict_proba(X_train_s))
    mlflow.sklearn.log_model(xgb,"model",signature=sig)
    print(classification_report(y_test,xgb.predict_proba(X_test_s).argmax(1),
                                  target_names=CLASS_NAMES))
    XGB_RUN=run.info.run_id
print(f"XGB mc  val_auc={vm['val_auc_macro']}  test_auc={tm['test_auc_macro']}")

In [ ]:
# ── CELL 12 : MODEL 3 — LightGBM multiclass ───────────────────────────────────
with mlflow.start_run(run_name="lightgbm_multiclass") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"lightgbm_multiclass"})
    lgb=LGBMClassifier(
        n_estimators=500,num_leaves=63,learning_rate=0.05,
        subsample=0.8,colsample_bytree=0.8,
        objective="multiclass",num_class=NUM_CLASSES,
        class_weight="balanced",n_jobs=-1,random_state=42,verbose=-1)
    lgb.fit(X_train_s,y_train,eval_set=[(X_val_s,y_val)],
            callbacks=[early_stopping(40,verbose=False),log_evaluation(100)])
    vm=evaluate_mc(lgb,X_val_s,y_val,"val"); tm=evaluate_mc(lgb,X_test_s,y_test,"test")
    mlflow.log_params({"n_estimators":500,"num_class":NUM_CLASSES,
                       "best_iter":lgb.best_iteration_})
    mlflow.log_metrics({**vm,**tm}); log_cm_mc(lgb,X_test_s,y_test,"lgb_mc")
    sig=infer_signature(X_train_s,lgb.predict_proba(X_train_s))
    mlflow.sklearn.log_model(lgb,"model",signature=sig); LGB_RUN=run.info.run_id
print(f"LGB mc  val_auc={vm['val_auc_macro']}  test_auc={tm['test_auc_macro']}")

In [ ]:
# ── CELL 13 : MODEL 4 — CatBoost multiclass ──────────────────────────────────
cat_idx=[i for i,c in enumerate(STRUCT_COLS) if "device_type" in c or "enc" in c]
with mlflow.start_run(run_name="catboost_multiclass") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"catboost_multiclass"})
    cb=CatBoostClassifier(
        iterations=500,depth=7,learning_rate=0.05,
        loss_function="MultiClass",eval_metric="TotalF1",
        class_weights=[1.0,1.5,2.0],  # upweight CRITICAL
        early_stopping_rounds=40,random_seed=42,task_type="CPU",verbose=100)
    cb.fit(X_train_s,y_train,eval_set=(X_val_s,y_val),
           cat_features=cat_idx if cat_idx else None)
    vm=evaluate_mc(cb,X_val_s,y_val,"val"); tm=evaluate_mc(cb,X_test_s,y_test,"test")
    mlflow.log_params({"iterations":500,"class_weights":"[1,1.5,2]",
                       "best_iter":cb.best_iteration_})
    mlflow.log_metrics({**vm,**tm}); log_cm_mc(cb,X_test_s,y_test,"cb_mc")
    cb.save_model("/tmp/ps3_catboost.cbm")
    mlflow.log_artifact("/tmp/ps3_catboost.cbm","catboost_native")
    sig=infer_signature(X_train_s,cb.predict_proba(X_train_s))
    mlflow.sklearn.log_model(cb,"model",signature=sig); CB_RUN=run.info.run_id
print(f"CB mc   val_auc={vm['val_auc_macro']}  test_auc={tm['test_auc_macro']}")

In [ ]:
# ── CELL 14 : MODEL 5 — Combined NLP + Structured (late fusion) ───────────────
# Only if text columns are available (PS3 70% feasibility scenario — updated 2026-06-23)
# TF-IDF on text → truncated SVD → concatenate with structured features → XGBoost

if text_cols and "combined_text" in df.columns:
    from sklearn.decomposition import TruncatedSVD
    from scipy.sparse import hstack, csr_matrix

    tfidf = TfidfVectorizer(max_features=3000,ngram_range=(1,2),
                            min_df=3,sublinear_tf=True)
    X_text_train = tfidf.fit_transform(train_df["combined_text"])
    X_text_val   = tfidf.transform(val_df["combined_text"])
    X_text_test  = tfidf.transform(test_df["combined_text"])

    # Reduce text to 50 dense components
    svd = TruncatedSVD(n_components=50, random_state=42)
    X_text_train_d = svd.fit_transform(X_text_train)
    X_text_val_d   = svd.transform(X_text_val)
    X_text_test_d  = svd.transform(X_text_test)

    # Late fusion: concatenate text components + structured features
    X_fused_train = np.hstack([X_train_s, X_text_train_d])
    X_fused_val   = np.hstack([X_val_s,   X_text_val_d])
    X_fused_test  = np.hstack([X_test_s,  X_text_test_d])
    X_fused_cv    = np.vstack([X_fused_train, X_fused_val])

    with mlflow.start_run(run_name="combined_nlp_structured") as run:
        mlflow.set_tags({**COMMON_TAGS,"model_type":"combined_nlp_structured"})
        mlflow.log_params({"text_components":50,"struct_features":len(STRUCT_COLS),
                           "total_features":X_fused_train.shape[1]})

        xgb_fused = XGBClassifier(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            objective="multi:softprob", num_class=NUM_CLASSES,
            eval_metric="mlogloss", early_stopping_rounds=30,
            tree_method="hist", random_state=42, n_jobs=-1)
        xgb_fused.fit(X_fused_train,y_train,
                       eval_set=[(X_fused_val,y_val)],verbose=100)

        vm=evaluate_mc(xgb_fused,X_fused_val,y_val,"val")
        tm=evaluate_mc(xgb_fused,X_fused_test,y_test,"test")
        mlflow.log_metrics({**vm,**tm})
        log_cm_mc(xgb_fused,X_fused_test,y_test,"combined_nlp")

        sig=infer_signature(X_fused_train,xgb_fused.predict_proba(X_fused_train))
        mlflow.sklearn.log_model(xgb_fused,"model",signature=sig)
        COMBINED_RUN=run.info.run_id

    print(f"Combined NLP+Struct  val_auc={vm['val_auc_macro']}  test_auc={tm['test_auc_macro']}")
    BEST_MODEL      = xgb_fused
    BEST_FEATURE_SET= "fused"
else:
    BEST_MODEL      = cb
    BEST_FEATURE_SET= "structured"
    COMBINED_RUN    = None

In [ ]:
# ── CELL 15 : Optuna — tune best model (all-fold) ────────────────────────────
X_opt = X_fused_cv if BEST_FEATURE_SET=="fused" else X_cv
TS_CV = TimeSeriesSplit(n_splits=5)

def optuna_obj(trial):
    p=dict(
        n_estimators    =trial.suggest_int("n_estimators",200,600),
        max_depth       =trial.suggest_int("max_depth",4,8),
        learning_rate   =trial.suggest_float("learning_rate",0.01,0.15,log=True),
        subsample       =trial.suggest_float("subsample",0.6,1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree",0.6,1.0),
        objective="multi:softprob", num_class=NUM_CLASSES,
        tree_method="hist", random_state=42, n_jobs=-1,
    )
    scores=[]
    for step,(tr,va) in enumerate(TS_CV.split(X_opt)):
        m=XGBClassifier(**p)
        m.fit(X_opt[tr],y_cv[tr],verbose=False)
        try:
            s=roc_auc_score(y_cv[va],m.predict_proba(X_opt[va]),
                            multi_class="ovr",average="macro")
        except: s=0.5
        scores.append(s)
        trial.report(np.mean(scores),step)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(scores)

study=optuna.create_study(direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5))
study.optimize(optuna_obj,n_trials=30,show_progress_bar=True)
BEST_PARAMS={**study.best_params,"objective":"multi:softprob","num_class":NUM_CLASSES,
             "tree_method":"hist","random_state":42,"n_jobs":-1}

with mlflow.start_run(run_name="xgboost_optuna_mc") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"xgboost_optuna_multiclass"})
    mlflow.log_params({**BEST_PARAMS,"optuna_trials":30,"optuna_cv":round(study.best_value,4)})
    xgb_tuned=XGBClassifier(**BEST_PARAMS)
    xgb_tuned.fit(X_opt[:len(X_fused_train)] if BEST_FEATURE_SET=="fused" else X_train_s,
                   y_train, verbose=False)
    X_test_use = X_fused_test if BEST_FEATURE_SET=="fused" else X_test_s
    X_val_use  = X_fused_val  if BEST_FEATURE_SET=="fused" else X_val_s
    vm=evaluate_mc(xgb_tuned,X_val_use,y_val,"val")
    tm=evaluate_mc(xgb_tuned,X_test_use,y_test,"test")
    mlflow.log_metrics({**vm,**tm}); log_cm_mc(xgb_tuned,X_test_use,y_test,"xgb_optuna_mc")
    X_sig = X_opt[:min(200, len(X_opt))]
    sig   = infer_signature(X_sig, xgb_tuned.predict_proba(X_sig))
    mlflow.sklearn.log_model(xgb_tuned,"model",signature=sig)
    XGB_TUNED_RUN=run.info.run_id
print(f"XGB(Optuna) mc  val_auc={vm['val_auc_macro']}  test_auc={tm['test_auc_macro']}")

In [ ]:
# ── CELL 16 : SHAP ────────────────────────────────────────────────────────────
explainer=shap.TreeExplainer(xgb_tuned)
X_shap=X_test_s[:1000]
sv=explainer(X_shap)

with mlflow.start_run(run_name="shap_analysis") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"shap_analysis"})
    shap.plots.bar(sv,max_display=20,show=False)
    fig = plt.gcf(); ax = plt.gca()
    ax.set_title("PS3 XGBoost — SHAP Feature Importance (multi-class)",pad=12)
    fig.tight_layout(); fig.savefig("/tmp/ps3_shap.png",dpi=100); plt.close(fig)
    mlflow.log_artifact("/tmp/ps3_shap.png")

    # Per-class SHAP
    if hasattr(sv,"values") and sv.values.ndim==3:
        for cls_idx,cls_name in enumerate(CLASS_NAMES):
            shap.plots.bar(sv[:,:,cls_idx],max_display=15,show=False)
            fig = plt.gcf(); ax = plt.gca()
            ax.set_title(f"PS3 SHAP — {cls_name} class",pad=12)
            fig.tight_layout()
            fig.savefig(f"/tmp/ps3_shap_{cls_name}.png",dpi=80); plt.close(fig)
            mlflow.log_artifact(f"/tmp/ps3_shap_{cls_name}.png")

In [ ]:
# ── CELL 17 : Select champion + register ─────────────────────────────────────
all_runs=mlflow.search_runs(experiment_names=[EXPERIMENT_NAME],
                             order_by=["metrics.test_auc_macro DESC"])
all_runs=all_runs[all_runs["tags.mlflow.runName"].str.contains(
    "xgboost|lightgbm|catboost|combined",na=False)]
CHAMPION_RUN_ID = all_runs.iloc[0]["run_id"]
CHAMPION_NAME   = all_runs.iloc[0]["tags.mlflow.runName"]
CHAMPION_AUC    = all_runs.iloc[0]["metrics.test_auc_macro"]
print(f"Champion: {CHAMPION_NAME}  AUC(macro)={CHAMPION_AUC:.4f}")

mlfc=mlflow.tracking.MlflowClient()
try: mlfc.create_registered_model(MLFLOW_MODEL_NAME,
        description="PS3 Root Cause Classification 3-class — Chicago Ventra")
except: pass
mv=mlfc.create_model_version(name=MLFLOW_MODEL_NAME,
    source=f"runs:/{CHAMPION_RUN_ID}/model",run_id=CHAMPION_RUN_ID,
    tags={"algorithm":CHAMPION_NAME,"test_auc_macro":str(CHAMPION_AUC),
          "feasibility":"70pct","target":"failure_level_label(3-class)"})
mlfc.transition_model_version_stage(MLFLOW_MODEL_NAME,mv.version,"Staging")
MLFLOW_VERSION=mv.version
print(f"MLflow: {MLFLOW_MODEL_NAME} v{MLFLOW_VERSION} → Staging")

# SageMaker registration
champion_model=mlflow.sklearn.load_model(f"runs:/{CHAMPION_RUN_ID}/model")
joblib.dump(champion_model,"/tmp/ps3_champion.joblib")
joblib.dump(STRUCT_COLS,"/tmp/ps3_feature_cols.joblib")
test_metrics={"test_auc_macro":CHAMPION_AUC}
S3_CLIENT.put_object(Bucket=ARTIFACT_BUCKET,Key="sagemaker/ps3/metrics/test_metrics.json",
                     Body=json.dumps(test_metrics),ContentType="application/json")
with tarfile.open("/tmp/model.tar.gz","w:gz") as tar:
    tar.add("/tmp/ps3_champion.joblib",arcname="ps3_champion.joblib")
    tar.add("/tmp/ps3_feature_cols.joblib",arcname="ps3_feature_cols.joblib")
MODEL_S3_KEY=f"sagemaker/ps3/model-v{MLFLOW_VERSION}/model.tar.gz"
S3_CLIENT.upload_file("/tmp/model.tar.gz",ARTIFACT_BUCKET,MODEL_S3_KEY)

import sagemaker.image_uris as sm_images
image_uri=sm_images.retrieve(
    framework="sklearn", region=REGION, version="1.2-1",
    instance_type="ml.m5.large", image_scope="inference")
try: SM_CLIENT.create_model_package_group(ModelPackageGroupName=MODEL_REGISTRY_NAME,
        ModelPackageGroupDescription="PS3 Root Cause — Chicago Ventra")
except: pass
resp=SM_CLIENT.create_model_package(
    ModelPackageGroupName=MODEL_REGISTRY_NAME,
    ModelPackageDescription=f"{CHAMPION_NAME}|AUC={CHAMPION_AUC:.4f}|Gv{GOLD_VERSION}",
    InferenceSpecification={
        "Containers":[{"Image":image_uri,"ModelDataUrl":f"s3://{ARTIFACT_BUCKET}/{MODEL_S3_KEY}"}],
        "SupportedContentTypes":["text/csv"],"SupportedResponseMIMETypes":["application/json"],
        "SupportedRealtimeInferenceInstanceTypes":["ml.m5.large"]},
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties={"ps":"PS3","algorithm":CHAMPION_NAME,
        "test_auc_macro":str(round(CHAMPION_AUC,4)),"feasibility":"70pct",
        "mlflow_version":str(MLFLOW_VERSION)})
SM_MODEL_PKG_ARN=resp["ModelPackageArn"]
print(f"SM Package: {SM_MODEL_PKG_ARN}")

In [ ]:
# ── CELL 18 : Deploy + Monitors ──────────────────────────────────────────────
from sagemaker.model import ModelPackage
from sagemaker.model_monitor import (DefaultModelMonitor,DataCaptureConfig,
                                      CronExpressionGenerator)
from sagemaker.model_monitor.dataset_format import DatasetFormat

SM_CLIENT.update_model_package(ModelPackageArn=SM_MODEL_PKG_ARN,
                                ModelApprovalStatus="Approved")
registered=ModelPackage(role=ROLE,model_package_arn=SM_MODEL_PKG_ARN,
                         sagemaker_session=SM_SESSION)
predictor=registered.deploy(
    initial_instance_count=1,instance_type="ml.m5.large",
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=DataCaptureConfig(
        enable_capture=True,sampling_percentage=100,
        destination_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps3/data-capture/",
        capture_options=["Input","Output"]))
print(f"Endpoint: {ENDPOINT_NAME}")

monitor=DefaultModelMonitor(role=ROLE,instance_count=1,
    instance_type="ml.m5.large",sagemaker_session=SM_SESSION)
pd.DataFrame(X_train_s,columns=STRUCT_COLS).to_csv("/tmp/ps3_baseline.csv",index=False)
S3_CLIENT.upload_file("/tmp/ps3_baseline.csv",ARTIFACT_BUCKET,
                      "sagemaker/ps3/baseline/baseline.csv")
monitor.suggest_baseline(
    baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps3/baseline/baseline.csv",
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps3/baseline/output/",
    wait=True,logs=False)
monitor.create_monitoring_schedule(
    monitor_schedule_name="chicago-ps3-data-quality",endpoint_input=ENDPOINT_NAME,
    output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps3/monitor-output/",
    statistics=monitor.baseline_statistics(),constraints=monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.daily(),
    enable_cloudwatch_metrics=True)
print("Monitor scheduled.")

In [ ]:
# ── CELL 19 : Summary ────────────────────────────────────────────────────────
print("="*60)
print("PS3 — Root Cause Classification — Complete")
print("="*60)
print(f"Champion     : {CHAMPION_NAME}  AUC(macro)={CHAMPION_AUC:.4f}")
print(f"MLflow model : {MLFLOW_MODEL_NAME} v{MLFLOW_VERSION}")
print(f"SM endpoint  : {ENDPOINT_NAME}")
print(f"Feature Store: {FS_GROUP_NAME}")
print(f"\nFEASIBILITY = 70%  (updated 2026-06-23, was 55%):")
print("  CONFIRMED: AE_FAILURE_LEVEL Ventra taxonomy (hardware faults, is_device_fault=TRUE):")
print("    1=NONPAYMENT  2=PURCHASE_CARD  3=PURCHASE_PRODUCT")
print("    4=ALL_PURCHASE  5=ALL_FUNCTIONS  16=BUS_READER_ASSEMBLY")
print("  Decoded via silver.dim_failure_level (S18).")
print("")
print("  REMAINING GAP (30%):")
print("  BLOCKED: SVN_STAGE.U_FS_FAULT_CODES (0 rows) → fc_description 9-class BLOCKED")
print("  BLOCKED: SVN_STAGE.U_FS_ACTION_CODES (0 rows) → ac_description 6-class BLOCKED")
print("  WORKAROUND: CTA.SERVICENOW_AVAILABILITY_EVENTS → 3-class failure_level_label ✓")
print("  WORKAROUND: Rule-based root_cause_category (8-class text match) ✓")
print("\nTo reach 100%: SVN_STAGE ETL must be activated (all 35 tables have 0 rows)")